<a href="https://colab.research.google.com/github/dodi-ctrl/PhishingDetector/blob/main/DistilBERT_Phishing_Text_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing the libraries
!pip install -q transformers datasets torch scikit-learn pandas accelerate
print("Installation OK")

Installation OK


In [ ]:
# Environmental assessment
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
import warnings
warnings.filterwarnings('ignore')

# Verify GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device detected: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("All good, we can proceed.")
else:
    print("NO GPU. Go to Runtime -> Change runtime type -> T4 GPU, then restart.")

Device detected: cuda
GPU: Tesla T4
All good, we can proceed.


In [ ]:
# Load the phishing email dataset from Hugging Face
print("Loading dataset")
dataset = load_dataset("zefang-liu/phishing-email-dataset")
df = pd.DataFrame(dataset['train'])

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3))
print(f"\nLabel distribution:")
for col in df.columns:
    if df[col].nunique() < 5:
        print(f"  {col}: {df[col].value_counts().to_dict()}")

Loading dataset


README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv:   0%|          | 0.00/52.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]


Dataset shape: (18650, 3)
Columns: ['Unnamed: 0', 'Email Text', 'Email Type']

First 3 rows:
   Unnamed: 0                                         Email Text  Email Type
0           0  re : 6 . 1100 , disc : uniformitarianism , re ...  Safe Email
1           1  the other side of * galicismos * * galicismo *...  Safe Email
2           2  re : equistar deal tickets are you still avail...  Safe Email

Label distribution:
  Email Type: {'Safe Email': 11322, 'Phishing Email': 7328}


In [ ]:
# Preprocessing and label encoding
TEXT_COL = 'Email Text'
LABEL_COL = 'Email Type'

# Encode labels: Safe Email -> 0, Phishing Email -> 1
df['label'] = df[LABEL_COL].apply(lambda x: 1 if 'Phishing' in str(x) else 0)

# Drop empty/null email texts
df = df.dropna(subset=[TEXT_COL])
df = df[df[TEXT_COL].astype(str).str.strip() != '']

# Keep only relevant columns and rename
df = df[[TEXT_COL, 'label']].rename(columns={TEXT_COL: 'text'}).reset_index(drop=True)

print(f"Total emails after cleaning: {len(df)}")
print(f"Label distribution:")
print(f"  Safe (0):     {(df['label'] == 0).sum()}")
print(f"  Phishing (1): {(df['label'] == 1).sum()}")

# Quick stats on email lengths
text_lens = df['text'].str.len()
print(f"\nText length stats:")
print(f"  Mean: {text_lens.mean():.0f} chars | Median: {text_lens.median():.0f} | Max: {text_lens.max()}")

Total emails after cleaning: 18631
Label distribution:
  Safe (0):     11322
  Phishing (1): 7309

Text length stats:
  Mean: 2756 chars | Median: 882 | Max: 17036692


In [ ]:
# Stratified 80/20 split (keeps the same label ratio in both sets)
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print(f"Training set: {len(train_df)} emails")
print(f"Test set:     {len(test_df)} emails")
print(f"\nTraining set label balance:")
print(f"  Safe:     {(train_df['label'] == 0).sum()} ({(train_df['label'] == 0).mean()*100:.1f}%)")
print(f"  Phishing: {(train_df['label'] == 1).sum()} ({(train_df['label'] == 1).mean()*100:.1f}%)")

Training set: 14904 emails
Test set:     3727 emails

Training set label balance:
  Safe:     9057 (60.8%)
  Phishing: 5847 (39.2%)


In [ ]:
# Load DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding=False,
        truncation=True,
        max_length=256,
    )

# Convert pandas DataFrames to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize both datasets
print("Tokenizing training set...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing test set...")
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f"\nDone. Train: {len(train_dataset)} | Test: {len(test_dataset)}")
print(f"Tokenized features: {list(train_dataset.features.keys())}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing training set...


Map:   0%|          | 0/14904 [00:00<?, ? examples/s]

Tokenizing test set...


Map:   0%|          | 0/3727 [00:00<?, ? examples/s]


Done. Train: 14904 | Test: 3727
Tokenized features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# Load DistilBERT for binary classification (Safe vs Phishing)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Safe", 1: "Phishing"},
    label2id={"Safe": 0, "Phishing": 1}
)

# Training hyperparameters
training_args = TrainingArguments(
    output_dir="./distilbert-phishing",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    fp16=True,
)

# Function called after each epoch to compute evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    accuracy = accuracy_score(labels, predictions)
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Data collator handles padding within each batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Build the Trainer (new API uses processing_class instead of tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Training setup ready.")
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training setup ready.
Model: distilbert-base-uncased
Epochs: 3
Batch size: 16
Learning rate: 2e-05


In [ ]:
# Start training
print("Starting training...\n")
trainer.train()
print("\nTraining complete.")

Starting training...



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.062430,0.091326,0.970754,0.949502,0.977428,0.963263
2,0.061323,0.084495,0.973169,0.952191,0.980848,0.966307
3,0.031745,0.101854,0.973437,0.955853,0.977428,0.966520


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training complete.


In [10]:
# Detailed evaluation on the test set
print("=" * 60)
print("FINAL EVALUATION ON TEST SET")
print("=" * 60)

# Get predictions on the test set
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["Safe Email", "Phishing Email"],
    digits=4
))

# Main metrics
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')

print(f"\nKey Metrics:")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1 Score : {f1:.4f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print(f"\nConfusion Matrix:")
print(f"                    Predicted Safe   Predicted Phishing")
print(f"  Actual Safe        {cm[0][0]:>10}        {cm[0][1]:>10}")
print(f"  Actual Phishing    {cm[1][0]:>10}        {cm[1][1]:>10}")

# Error analysis
total_safe = cm[0][0] + cm[0][1]
total_phishing = cm[1][0] + cm[1][1]
fpr = cm[0][1] / total_safe if total_safe else 0
fnr = cm[1][0] / total_phishing if total_phishing else 0

print(f"\nError Analysis:")
print(f"  False Positive Rate (Safe flagged as Phishing): {fpr:.4f} ({cm[0][1]} / {total_safe})")
print(f"  False Negative Rate (Phishing missed):           {fnr:.4f} ({cm[1][0]} / {total_phishing})")

FINAL EVALUATION ON TEST SET



Classification Report:
                precision    recall  f1-score   support

    Safe Email     0.9852    0.9709    0.9780      2265
Phishing Email     0.9559    0.9774    0.9665      1462

      accuracy                         0.9734      3727
     macro avg     0.9705    0.9741    0.9723      3727
  weighted avg     0.9737    0.9734    0.9735      3727


Key Metrics:
  Accuracy : 0.9734
  Precision: 0.9559
  Recall   : 0.9774
  F1 Score : 0.9665

Confusion Matrix:
                    Predicted Safe   Predicted Phishing
  Actual Safe              2199                66
  Actual Phishing            33              1429

Error Analysis:
  False Positive Rate (Safe flagged as Phishing): 0.0291 (66 / 2265)
  False Negative Rate (Phishing missed):           0.0226 (33 / 1462)


In [11]:
# Test the trained model on real-world email examples
def predict_email(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred_class = torch.argmax(probs).item()
    label = "PHISHING" if pred_class == 1 else "SAFE"
    confidence = probs[pred_class].item()
    return label, confidence, probs

# Phishing example (typical: urgency + suspicious URL + threats)
phishing_example = """
Dear Customer,
URGENT: Your account has been temporarily suspended due to suspicious activity.
You must verify your identity within 24 hours to avoid permanent deletion.
Click here immediately: http://secure-bank-verify.tk/login?id=98234
Failure to act will result in loss of access.
Bank Security Team
"""

# Legitimate workplace email
legit_example = """
Hi team,
Just a reminder that we have our weekly standup tomorrow at 10am in Conference Room B.
The agenda is in the shared drive under "Q2 Planning".
Let me know if you have any items to add.
Thanks,
Sarah
"""

# Borderline case (file sharing - could be either)
ambiguous_example = """
Hello,
I've shared the document we discussed last week. You can access it here:
https://drive.google.com/file/d/1abc123/view
Please review and send me your feedback by Friday.
Best,
Mike
"""

print("=" * 60)
print("REAL-WORLD EMAIL TESTS")
print("=" * 60)

for name, email in [
    ("Phishing example (urgent + suspicious URL)", phishing_example),
    ("Legitimate email (workplace meeting)", legit_example),
    ("Borderline case (file sharing)", ambiguous_example),
]:
    label, conf, probs = predict_email(email, model, tokenizer)
    print(f"\n>>> {name}")
    print(f"    Prediction: {label} (confidence: {conf:.2%})")
    print(f"    Safe probability:     {probs[0].item():.4f}")
    print(f"    Phishing probability: {probs[1].item():.4f}")

REAL-WORLD EMAIL TESTS

>>> Phishing example (urgent + suspicious URL)
    Prediction: PHISHING (confidence: 99.93%)
    Safe probability:     0.0007
    Phishing probability: 0.9993

>>> Legitimate email (workplace meeting)
    Prediction: SAFE (confidence: 99.99%)
    Safe probability:     0.9999
    Phishing probability: 0.0001

>>> Borderline case (file sharing)
    Prediction: SAFE (confidence: 99.99%)
    Safe probability:     0.9999
    Phishing probability: 0.0001


In [12]:
import shutil
import os

OUTPUT_DIR = "./distilbert_phishing_text_agent"

# Save model + tokenizer to disk
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Show what was saved
print("Files saved in", OUTPUT_DIR + ":")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

# Create a zip archive for easy download / sharing
print("\nCreating zip archive...")
shutil.make_archive("distilbert_phishing_text_agent", 'zip', OUTPUT_DIR)
zip_size = os.path.getsize("distilbert_phishing_text_agent.zip") / 1024 / 1024
print(f"distilbert_phishing_text_agent.zip ({zip_size:.1f} MB)")

# Trigger download to your laptop
from google.colab import files
files.download("distilbert_phishing_text_agent.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Files saved in ./distilbert_phishing_text_agent:
  config.json (0.0 MB)
  model.safetensors (255.4 MB)
  tokenizer.json (0.7 MB)
  tokenizer_config.json (0.0 MB)
  training_args.bin (0.0 MB)

Creating zip archive...
distilbert_phishing_text_agent.zip (235.8 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>